In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pylab as plt
import pickle
import sklearn
from sklearn import metrics

from sklearn.manifold import TSNE
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, accuracy_score, recall_score, roc_auc_score

RandomSeed = 42
np.random.seed(RandomSeed)

sns.set(font_scale = 1.4)
sns.set_style("white")
sns.set_style("ticks", {"xtick.major.size": 6, "ytick.major.size": 0})

pd.set_option("display.max_colwidth", False)
pd.set_option('display.expand_frame_repr', False)
sns.set(font_scale = 1.2)

In [51]:
import pandas as pd

#DO NOT CHANGE ANYTHING IN THIS CELL. MOVE ON TO THE FOLLOWING ONE TO GET THE PREDICTION.
import os
os.environ['PATH'] = "/Users/newuser/ncbi-blast-2.16.0+/bin:" + os.environ['PATH']
os.environ['PATH'] = "C:/Program Files/NCBI/blast-2.16.0+/bin" + ";" + os.environ['PATH']


import numpy as np
from gensim.models import word2vec

class ProtVec(word2vec.Word2Vec):

    def __init__(self, fasta_fname=None, corpus=None, n=3, size=100, corpus_fname="corpus.txt",  sg=1, window=25, min_count=1, workers=20):
        """
        Either fname or corpus is required.
        fasta_fname: fasta file for corpus
        corpus: corpus object implemented by gensim
        n: n of n-gram
        corpus_fname: corpus file path
        min_count: least appearance count in corpus. if the n-gram appear k times which is below min_count, the model does not remember the n-gram
        """

        self.n = n
        self.size = size
        self.fasta_fname = fasta_fname

        if corpus is None and fasta_fname is None:
            raise Exception("Either fasta_fname or corpus is needed!")

        if fasta_fname is not None:
            print('Generate Corpus file from fasta file...')
            generate_corpusfile(fasta_fname, n, corpus_fname)
            corpus = word2vec.Text8Corpus(corpus_fname)

        word2vec.Word2Vec.__init__(self, corpus, size=size, sg=sg, window=window, min_count=min_count, workers=workers)

    def to_vecs(self, seq):
        """
        convert sequence to three n-length vectors
        e.g. 'AGAMQSASM' => [ array([  ... * 100 ], array([  ... * 100 ], array([  ... * 100 ] ]
        """
        ngram_patterns = split_ngrams(seq, self.n)

        protvecs = []
        for ngrams in ngram_patterns:
            ngram_vecs = []
            for ngram in ngrams:
                try:
                    ngram_vecs.append(self.wv[ngram])
                except:
                    raise Exception("Model has never trained this n-gram: " + ngram)
            protvecs.append(sum(ngram_vecs))
        return protvecs
    
    
    def get_vector(self, seq):
        """
        sum and normalize the three n-length vectors returned by self.to_vecs
        """
        #return normalize(sum(self.to_vecs(seq)))
        return sum(self.to_vecs(seq))

    
def load_protvec(model_fname):
    return word2vec.Word2Vec.load(model_fname)

pv = load_protvec('src/files/DeePhase/__PREDICT/tools/Embeddings/swissprot_size200_window25.model')

SEED = 42
np.random.seed(SEED)

from src.files.DeePhase.__PREDICT.deephase_utils import *

def extract_seq_feature(seq):
    # # Create a DataFrame with the input sequence
    df = pd.DataFrame({'sequence_final': [seq]})
    
    data_interm = create_features(df)

    return data_interm

def extract_seq_deephase_score(seq):
    # # Create a DataFrame with the input sequence
    df = pd.DataFrame({'sequence_final': [seq]})
    
    # Call the DeePhase function (assuming it returns a string)
    deephase_result = DeePhase(df)

    df_deephase_score = pd.DataFrame([deephase_result], columns=['deephase_phys_multi', 'deephase_w2v_multi', 'deephase_score'])

    df_deephase_score = df_deephase_score.astype(float)

    return df_deephase_score

In [52]:
sequence = "HTWDAAAAAAAAAAAAGAWEWSIDTEAGGGRREQSQKPCSNGGPAAAGEGRVLPSPCFPWSTCQAAIHKVCRWQGCTRPALLAPSLATLKEHSYP"
# extract_seq_deephase_score(sequence)
extracted_feature_df = extract_seq_feature(sequence)

In [53]:
list(extracted_feature_df.columns)

['sequence_final',
 'Sequence_length',
 'LCR_frac',
 'LCR_length',
 'Hydrophobicity',
 'Shannon_entropy',
 'IDR_frac',
 'pI',
 'AA_A',
 'AA_C',
 'AA_D',
 'AA_E',
 'AA_F',
 'AA_G',
 'AA_H',
 'AA_I',
 'AA_K',
 'AA_L',
 'AA_M',
 'AA_N',
 'AA_P',
 'AA_Q',
 'AA_R',
 'AA_S',
 'AA_T',
 'AA_V',
 'AA_W',
 'AA_Y',
 'Polar',
 'Cation',
 'Anion',
 'Arom',
 'HB',
 'HB_frac',
 'Polar_frac',
 'Arom_frac',
 'Cation_frac',
 'Anion_frac',
 'AA_LCR_A',
 'AA_LCR_C',
 'AA_LCR_D',
 'AA_LCR_E',
 'AA_LCR_F',
 'AA_LCR_G',
 'AA_LCR_H',
 'AA_LCR_I',
 'AA_LCR_K',
 'AA_LCR_L',
 'AA_LCR_M',
 'AA_LCR_N',
 'AA_LCR_P',
 'AA_LCR_Q',
 'AA_LCR_R',
 'AA_LCR_S',
 'AA_LCR_T',
 'AA_LCR_V',
 'AA_LCR_W',
 'AA_LCR_Y',
 'Polar_LCR',
 'Cation_LCR',
 'Anion_LCR',
 'Arom_LCR',
 'HB_LCR',
 'HB_LCR_frac',
 'Polar_LCR_frac',
 'Arom_LCR_frac',
 'Cation_LCR_frac',
 'Anion_LCR_frac',
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 

In [54]:
combined = pd.read_csv('src/files/DeePhase/Paper_code/Features/training_data_features.csv')

In [55]:
combined.columns

Index(['Sequence', 'Uniprot_ID', 'Category', 'Sequence_length', 'LCR_frac',
       'LCR_sequence', 'LCR_length', 'Hydrophobicity', 'Shannon_entropy',
       'IDR_frac',
       ...
       '190', '191', '192', '193', '194', '195', '196', '197', '198', '199'],
      dtype='object', length=274)

In [56]:
def find_set_differences_and_common(list1, list2):
    # Convert all elements to strings
    set1 = set(str(item) for item in list1)
    set2 = set(str(item) for item in list2)
    
    # Elements in list1 but not in list2
    only_in_list1 = list(set1 - set2)
    
    # Elements in list2 but not in list1
    only_in_list2 = list(set2 - set1)
    
    # Elements in both lists
    in_both = list(set1 & set2)
    
    return only_in_list1, only_in_list2, in_both

# Example usage
list1 = list(extracted_feature_df.columns)
list2 = list(combined.columns)

unique_to_list1, unique_to_list2, common_elements = find_set_differences_and_common(list1, list2)

print("In list1 but not in list2:", unique_to_list1)
print("In list2 but not in list1:", unique_to_list2)
print("In both lists:", common_elements)

In list1 but not in list2: ['sequence_final']
In list2 but not in list1: ['reshuffled_sequence', 'delta_5_HB_RS', 'Uniprot_ID', 'delta_5_HB', 'LCR_sequence', 'Category', 'Sequence']
In both lists: ['AA_LCR_E', '161', '189', 'AA_G', 'Arom_LCR', '198', '122', '156', 'AA_LCR_G', '79', '174', 'Arom_frac', '179', 'AA_T', '133', '62', '139', 'AA_M', '53', '145', 'AA_LCR_W', '13', '39', 'Cation', '63', '160', '116', '7', 'AA_Q', 'AA_LCR_V', '77', '167', '54', '117', '91', '148', '75', '42', '45', '199', '108', 'Polar_LCR', '81', '130', '37', 'LCR_length', '158', '6', 'AA_L', '36', '66', '151', '19', '104', 'pI', '59', '47', '185', '152', '182', '115', '143', '165', '144', '83', 'AA_LCR_H', '190', 'AA_LCR_C', '57', 'Anion_frac', 'Anion_LCR_frac', '146', 'AA_E', '114', 'HB_LCR', '197', '10', '51', 'HB_frac', '35', '55', '103', '180', '86', '119', 'Cation_frac', 'AA_LCR_T', 'AA_LCR_R', '87', '74', '112', '164', '183', '172', 'AA_V', '34', '11', '147', '67', '141', '163', '88', '24', '38', 'Polar

In [57]:
# the column need to add to are
add_column = ['reshuffled_sequence', 'delta_5_HB_RS', 'Uniprot_ID', 'delta_5_HB', 'LCR_sequence', 'Category', 'Sequence']
info_cols = ['Uniprot_ID', 'Sequence', 'Category']

In [58]:
# download dataset from picnic
# https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-024-55089-x/MediaObjects/41467_2024_55089_MOESM2_ESM.xlsx

import pandas as pd
import requests
import os

# URL of the dataset
url = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-024-55089-x/MediaObjects/41467_2024_55089_MOESM2_ESM.xlsx"

def download_and_process_picnic_dataset():
    # Create a directory to save the file if it doesn't exist
    os.makedirs('data', exist_ok=True)
    
    # Define the local file path
    local_file = 'data/picnic_dataset.xlsx'
    
    # Check if file already exists
    if not os.path.exists(local_file):
        print(f"Downloading PICNIC dataset from {url}...")
        
        # Download the file
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for HTTP errors
        
        # Save the file locally
        with open(local_file, 'wb') as f:
            f.write(response.content)
        
        print(f"File downloaded and saved to {local_file}")
    else:
        print(f"File already exists at {local_file}")
    
    # Load the Excel file
    print("Loading the dataset...")
    
    # Get all sheet names
    xls = pd.ExcelFile(local_file)
    sheet_names = xls.sheet_names
    print(f"Available sheets: {sheet_names}")
    
    # Read the first sheet (LLPS+)
    df_positive = pd.read_excel(local_file, sheet_name=0)
    df_positive['Category'] = 'LLPS+'
    print(f"Loaded LLPS+ sheet with {df_positive.shape[0]} rows and {df_positive.shape[1]} columns")
    
    # Read the second sheet (LLPS-)
    df_negative = pd.read_excel(local_file, sheet_name=1)
    df_negative['Category'] = 'LLPS-'
    print(f"Loaded LLPS- sheet with {df_negative.shape[0]} rows and {df_negative.shape[1]} columns")
    
    # Combine the two dataframes
    combined_df = pd.concat([df_positive, df_negative], ignore_index=True)
    
    # Display information about the combined dataset
    print("\nCombined dataset information:")
    print(f"Shape: {combined_df.shape}")
    print("\nColumn names:")
    print(combined_df.columns.tolist())
    print("\nFirst few rows:")
    print(combined_df.head())
    
    # Show category distribution
    print("\nCategory distribution:")
    print(combined_df['Category'].value_counts())
    
    return combined_df, sheet_names
    
picnic_df, available_sheets = download_and_process_picnic_dataset()

File already exists at data/picnic_dataset.xlsx
Loading the dataset...
Available sheets: ['All(train+test)_positive', 'All(train+test)_negative', 'Test_positive', 'Test_negative', 'Test_negative_mlo', 'Test_positive_Opencell', 'Test_positive_PhasepHT']
Loaded LLPS+ sheet with 2142 rows and 2 columns
Loaded LLPS- sheet with 1709 rows and 2 columns

Combined dataset information:
Shape: (3851, 2)

Column names:
['uniprot_id', 'Category']

First few rows:
  uniprot_id Category
0  P62312     LLPS+  
1  Q3SY89     LLPS+  
2  Q9H898     LLPS+  
3  Q96PF2     LLPS+  
4  Q9BXY0     LLPS+  

Category distribution:
Category
LLPS+    2142
LLPS-    1709
Name: count, dtype: int64


In [59]:
picnic_df

,uniprot_id,Category
0,P62312,LLPS+
1,Q3SY89,LLPS+
2,Q9H898,LLPS+
3,Q96PF2,LLPS+
4,Q9BXY0,LLPS+
...,...,...
3846,Q96EY8,LLPS-
3847,Q96K19,LLPS-
3848,Q02080,LLPS-
3849,Q07654,LLPS-


In [60]:
import pandas as pd
import requests
import re
from io import StringIO
import time

def read_fasta_file(fasta_file):
    """Parse FASTA file into a dictionary with UniProt IDs as keys and sequences as values"""
    print(f"Reading FASTA file: {fasta_file}")
    fasta_dict = {}
    current_id = None
    current_sequence = []
    
    with open(fasta_file, 'r') as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
                
            if line.startswith('>'):
                # If we have a previous sequence, save it
                if current_id is not None:
                    fasta_dict[current_id] = ''.join(current_sequence)
                
                # Extract UniProt ID from the header
                # The pattern for UniProt IDs in the header is >sp|UNIPROT_ID|...
                match = re.search(r'>sp\|([A-Z0-9]+)\|', line)
                if match:
                    current_id = match.group(1)
                    current_sequence = []
            else:
                # Add sequence line
                current_sequence.append(line)
        
        # Add the last sequence
        if current_id is not None:
            fasta_dict[current_id] = ''.join(current_sequence)
    
    print(f"Parsed {len(fasta_dict)} sequences from FASTA file")
    return fasta_dict

def fetch_sequence_from_uniprot(uniprot_id):
    """Fetch protein sequence from UniProt API"""
    try:
        url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
        response = requests.get(url)
        
        if response.status_code == 200:
            fasta_content = response.text
            # Skip the header line and join the sequence lines
            sequence = ''.join(fasta_content.split('\n')[1:])
            return sequence
        else:
            print(f"Failed to fetch sequence for {uniprot_id}: HTTP {response.status_code}")
            return None
    except Exception as e:
        print(f"Error fetching sequence for {uniprot_id}: {str(e)}")
        return None

def add_sequences_to_df(df, fasta_file):
    """Add sequences to DataFrame based on UniProt IDs"""
    # Read sequences from FASTA file
    fasta_dict = read_fasta_file(fasta_file)
    
    # Initialize a new column for sequences
    df['sequence'] = None
    
    # Counter for API requests to avoid hitting rate limits
    api_request_count = 0
    
    for index, row in df.iterrows():
        uniprot_id = row['uniprot_id']
        
        # Check if ID exists in the FASTA dictionary
        if uniprot_id in fasta_dict:
            df.at[index, 'sequence'] = fasta_dict[uniprot_id]
        else:
            print(f"UniProt ID {uniprot_id} not found in FASTA file, fetching from UniProt...")
            
            # Add a delay every 10 requests to avoid rate limiting
            if api_request_count > 0 and api_request_count % 10 == 0:
                print("Pausing for 1 second to avoid rate limiting...")
                time.sleep(1)
            
            sequence = fetch_sequence_from_uniprot(uniprot_id)
            df.at[index, 'sequence'] = sequence
            api_request_count += 1
    
    # Count how many sequences were found vs. not found
    found_count = df['sequence'].notna().sum()
    not_found_count = df['sequence'].isna().sum()
    
    print(f"Added sequences to DataFrame: {found_count} found, {not_found_count} not found")
    
    return df


# FASTA file path
fasta_file = "uniprotkb_organism_id_9606_AND_reviewed_2025_02_19.fasta"

# Add sequences to DataFrame
df_with_sequences = add_sequences_to_df(picnic_df, fasta_file)

# Save the DataFrame with sequences to a CSV file
output_file = "protein_sequences.csv"
df_with_sequences.to_csv(output_file, index=False)
print(f"Saved DataFrame with sequences to {output_file}")

# Display the first few rows
print("\nFirst few rows of the DataFrame with sequences:")
print(df_with_sequences.head())


Reading FASTA file: uniprotkb_organism_id_9606_AND_reviewed_2025_02_19.fasta
Parsed 20417 sequences from FASTA file
UniProt ID Q3SY89 not found in FASTA file, fetching from UniProt...
UniProt ID Q6IE36 not found in FASTA file, fetching from UniProt...
Added sequences to DataFrame: 3851 found, 0 not found
Saved DataFrame with sequences to protein_sequences.csv

First few rows of the DataFrame with sequences:
  uniprot_id Category                                                                                                                                                                                                                                                                                                                                                                sequence
0  P62312     LLPS+    MSLRKQTPSDFLKQIIGRPVVVKLNSGVDYRGVLACLDGYMNIALEQTEEYVNGQLKNKYGDAFIRGNNVLYISTQKRRM                                                                                                        

In [61]:
df_with_sequences

,uniprot_id,Category,sequence
0,P62312,LLPS+,MSLRKQTPSDFLKQIIGRPVVVKLNSGVDYRGVLACLDGYMNIALEQTEEYVNGQLKNKYGDAFIRGNNVLYISTQKRRM
1,Q3SY89,LLPS+,
2,Q9H898,LLPS+,MKSSDIDQDLFTDSYCKVCSAQLISESQRVAHYESRKHASKVRLYYMLHPRDGGCPAKRLRSENGSDADMVDKNKCCTLCNMSFTSAVVADSHYQGKIHAKRLKLLLGEKTPLKTTATPLSPLKPPRMDTAPVVASPYQRRDSDRYCGLCAAWFNNPLMAQQHYDGKKHKKNAARVALLEQLGTTLDMGELRGLRRNYRCTICSVSLNSIEQYHAHLKGSKHQTNLKNK
3,Q96PF2,LLPS+,MDDATVLRKKGYIVGINLGKGSYAKVKSAYSERLKFNVAVKIIDRKKTPTDFVERFLPREMDILATVNHGSIIKTYEIFETSDGRIYIIMELGVQGDLLEFIKCQGALHEDVARKMFRQLSSAVKYCHDLDIVHRDLKCENLLLDKDFNIKLSDFGFSKRCLRDSNGRIILSKTFCGSAAYAAPEVLQSIPYQPKVYDIWSLGVILYIMVCGSMPYDDSDIRKMLRIQKEHRVDFPRSKNLTCECKDLIYRMLQPDVSQRLHIDEILSHSWLQPPKPKATSSASFKREGEGKYRAECKLDTKTGLRPDHRPDHKLGAKTQHRLLVVPENENRMEDRLAETSRAKDHHISGAEVGKAST
4,Q9BXY0,LLPS+,MQSDDVIWDTLGNKQFCSFKIRTKTQSFCRNEYSLTGLCNRSSCPLANSQYATIKEEKGQCYLYMKVIERAAFPRRLWERVRLSKNYEKALEQIDENLIYWPRFIRHKCKQRFTKITQYLIRIRKLTLKRQRKLVPLSKKVERREKRREEKALIAAQLDNAIEKELLERLKQDTYGDIYNFPIHAFDKALEQQEAESDSSDTEEKDDDDDDEEDVGKREFVEDGEVDESDISDFEDMDKLDASSDEDQDGKSSSEEEEEKALSAKHKGKMPLRGPLQRKRAYVEIEYEQETEPVAKAKTT
...,...,...,...
3846,Q96EY8,LLPS-,MAVCGLGSRLGLGSRLGLRGCFGAARLLYPRFQSRGPQGVEDGDRPQPSSKTPRIPKIYTKTGDKGFSSTFTGERRPKDDQVFEAVGTTDELSSAIGFALELVTEKGHTFAEELQKIQCTLQDVGSALATPCSSAREAHLKYTTFKAGPILELEQWIDKYTSQLPPLTAFILPSGGKISSALHFCRAVCRRAERRVVPLVQMGETDANVAKFLNRLSDYLFTLARYAAMKEGNQEKIYMKNDPSAESEGL
3847,Q96K19,LLPS-,MAKYQGEVQSLKLDDDSVIEGVSDQVLVAVVVSFALIATLVYALFRNVHQNIHPENQELVRVLREQLQTEQDAPAATRQQFYTDMYCPICLHQASFPVETNCGHLFCGACIIAYWRYGSWLGAISCPICRQTVTLLLTVFGEDDQSQDVLRLHQDINDYNRRFSGQPRSIMERIMDLPTLLRHAFREMFSVGGLFWMFRIRIILCLMGAFFYLISPLDFVPEALFGILGFLDDFFVIFLLLIYISIMYREVITQRLTR
3848,Q02080,LLPS-,MGRKKIQISRILDQRNRQVTFTKRKFGLMKKAYELSVLCDCEIALIIFNSANRLFQYASTDMDRVLLKYTEYSEPHESRTNTDILETLKRRGIGLDGPELEPDEGPEEPGEKFRRLAGEGGDPALPRPRLYPAAPAMPSPDVVYGALPPPGCDPSGLGEALPAQSRPSPFRPAAPKAGPPGLVHPLFSPSHLTSKTPPPLYLPTEGRRSDLPGGLAGPRGGLNTSRSLYSGLQNPCSTATPGPPLGSFPFLPGGPPVGAEAWARRVPQPAAPPRRPPQSASSLSASLRPPGAPATFLRPSPIPCSSPGPWQSLCGLGPPCAGCPWPTAGPGRRSPGGTSPERSPGTARARGDPTSLQASSEKTQQ
3849,Q07654,LLPS-,MAARALCMLGLVLALLSSSSAEEYVGLSANQCAVPAKDRVDCGYPHVTPKECNNRGCCFDSRIPGVPWCFKPLQEAECTF


In [62]:
def analyze_missing_sequences(df):
    """
    Analyzes a DataFrame to find entries without sequences
    Returns the count and list of UniProt IDs with missing sequences
    """
    # Find rows where sequence is None, NaN, or empty string
    missing_sequences = df[df['sequence'].isna() | (df['sequence'] == '')]
    
    # Count of missing sequences
    missing_count = len(missing_sequences)
    
    # List of UniProt IDs with missing sequences
    missing_ids = missing_sequences['uniprot_id'].tolist()
    
    print(f"Total proteins: {len(df)}")
    print(f"Missing sequences: {missing_count} ({missing_count/len(df)*100:.2f}%)")
    print("\nUniProt IDs with missing sequences:")
    
    # Print the IDs in a readable format
    if missing_ids:
        for uniprot_id in missing_ids:
            print(f"- {uniprot_id}")
    else:
        print("None - all sequences were found!")
    
    # Return for further processing if needed
    return missing_count, missing_ids

In [63]:
missing_count, missing_ids = analyze_missing_sequences(df_with_sequences)

Total proteins: 3851
Missing sequences: 2 (0.05%)

UniProt IDs with missing sequences:
- Q3SY89
- Q6IE36


In [64]:
def drop_missing_sequences(df):
    """
    Removes rows from DataFrame where sequence data is missing
    Returns the cleaned DataFrame and count of dropped rows
    """
    # Count rows before dropping
    original_count = len(df)
    
    # Drop rows where sequence is None, NaN, or empty string
    df_clean = df.dropna(subset=['sequence'])
    
    # Remove rows with empty strings as well (in case they're not caught by dropna)
    df_clean = df_clean[df_clean['sequence'] != '']
    
    # Count how many rows were dropped
    dropped_count = original_count - len(df_clean)
    
    print(f"Original dataset: {original_count} proteins")
    print(f"Dropped {dropped_count} proteins with missing sequences")
    print(f"Remaining dataset: {len(df_clean)} proteins")
    
    return df_clean
df_clean = drop_missing_sequences(df_with_sequences)
df_clean

Original dataset: 3851 proteins
Dropped 2 proteins with missing sequences
Remaining dataset: 3849 proteins


,uniprot_id,Category,sequence
0,P62312,LLPS+,MSLRKQTPSDFLKQIIGRPVVVKLNSGVDYRGVLACLDGYMNIALEQTEEYVNGQLKNKYGDAFIRGNNVLYISTQKRRM
2,Q9H898,LLPS+,MKSSDIDQDLFTDSYCKVCSAQLISESQRVAHYESRKHASKVRLYYMLHPRDGGCPAKRLRSENGSDADMVDKNKCCTLCNMSFTSAVVADSHYQGKIHAKRLKLLLGEKTPLKTTATPLSPLKPPRMDTAPVVASPYQRRDSDRYCGLCAAWFNNPLMAQQHYDGKKHKKNAARVALLEQLGTTLDMGELRGLRRNYRCTICSVSLNSIEQYHAHLKGSKHQTNLKNK
3,Q96PF2,LLPS+,MDDATVLRKKGYIVGINLGKGSYAKVKSAYSERLKFNVAVKIIDRKKTPTDFVERFLPREMDILATVNHGSIIKTYEIFETSDGRIYIIMELGVQGDLLEFIKCQGALHEDVARKMFRQLSSAVKYCHDLDIVHRDLKCENLLLDKDFNIKLSDFGFSKRCLRDSNGRIILSKTFCGSAAYAAPEVLQSIPYQPKVYDIWSLGVILYIMVCGSMPYDDSDIRKMLRIQKEHRVDFPRSKNLTCECKDLIYRMLQPDVSQRLHIDEILSHSWLQPPKPKATSSASFKREGEGKYRAECKLDTKTGLRPDHRPDHKLGAKTQHRLLVVPENENRMEDRLAETSRAKDHHISGAEVGKAST
4,Q9BXY0,LLPS+,MQSDDVIWDTLGNKQFCSFKIRTKTQSFCRNEYSLTGLCNRSSCPLANSQYATIKEEKGQCYLYMKVIERAAFPRRLWERVRLSKNYEKALEQIDENLIYWPRFIRHKCKQRFTKITQYLIRIRKLTLKRQRKLVPLSKKVERREKRREEKALIAAQLDNAIEKELLERLKQDTYGDIYNFPIHAFDKALEQQEAESDSSDTEEKDDDDDDEEDVGKREFVEDGEVDESDISDFEDMDKLDASSDEDQDGKSSSEEEEEKALSAKHKGKMPLRGPLQRKRAYVEIEYEQETEPVAKAKTT
5,Q14676,LLPS+,MEDTQAIDWDVEEEEETEQSSESLRCNVEPVGRLHIFSGAHGPEKDFPLHLGKNVVGRMPDCSVALPFPSISKQHAEIEILAWDKAPILRDCGSLNGTQILRPPKVLSPGVSHRLRDQELILFADLLCQYHRLDVSLPFVSRGPLTVEETPRVQGETQPQRLLLAEDSEEEVDFLSERRMVKKSRTTSSSVIVPESDEEGHSPVLGGLGPPFAFNLNSDTDVEEGQQPATEEASSAARRGATVEAKQSEAEVVTEIQLEKDQPLVKERDNDTKVKRGAGNGVVPAGVILERSQPPGEDSDTDVDDDSRPPGRPAEVHLERAQPFGFIDSDTDAEEERIPATPVVIPMKKRKIFHGVGTRGPGAPGLAHLQESQAGSDTDVEEGKAPQAVPLEKSQASMVINSDTDDEEEVSAALTLAHLKESQPAIWNRDAEEDMPQRVVLLQRSQTTTERDSDTDVEEEELPVENREAVLKDHTKIRALVRAHSEKDQPPFGDSDDSVEADKSSPGIHLERSQASTTVDINTQVEKEVPPGSAIIHIKKHQVSVEGTNQTDVKAVGGPAKLLVVSLEEAWPLHGDCETDAEEGTSLTASVVADVRKSQLPAEGDAGAEWAAAVLKQERAHEVGAQGGPPVAQVEQDLPISRENLTDLVVDTDTLGESTQPQREGAQVPTGREREQHVGGTKDSEDNYGDSEDLDLQATQCFLENQGLEAVQSMEDEPTQAFMLTPPQELGPSHCSFQTTGTLDEPWEVLATQPFCLRESEDSETQPFDTHLEAYGPCLSPPRAIPGDQHPESPVHTEPMGIQGRGRQTVDKVMGIPKETAERVGPERGPLERETEKLLPERQTDVTGEEELTKGKQDREQKQLLARDTQRQESDKNGESASPERDRESLKVEIETSEEIQEKQVQKQTLPSKAFEREVERPVANRECDPAELEEKVPKVILERDTQRGEPEGGSQDQKGQASSPTPEPGVGAGDLPGPTSAPVPSGSQSGGRGSPVSPRRHQKGLLNCKMPPAEKASRIRAAEKVSRGDQESPDACLPPTVPEAPAPPQKPLNSQSQKHLAPPPLLSPLLPSIKPTVRKTRQDGSQEAPEAPLSSELEPFHPKPKIRTRKSSRMTPFPATSAAPEPHPSTSTAQPVTPKPTSQATRSRTNRSSVKTPEPVVPTAPELQPSTSTDQPVTSEPTSQVTRGRKSRSSVKTPETVVPTALELQPSTSTDRPVTSEPTSQATRGRKNRSSVKTPEPVVPTAPELQPSTSTDQPVTSEPTYQATRGRKNRSSVKTPEPVVPTAPELRPSTSTDRPVTPKPTSRTTRSRTNMSSVKTPETVVPTAPELQISTSTDQPVTPKPTSRTTRSRTNMSSVKNPESTVPIAPELPPSTSTEQPVTPEPTSRATRGRKNRSSGKTPETLVPTAPKLEPSTSTDQPVTPEPTSQATRGRTNRSSVKTPETVVPTAPELQPSTSTDQPVTPEPTSQATRGRTDRSSVKTPETVVPTAPELQASASTDQPVTSEPTSRTTRGRKNRSSVKTPETVVPAAPELQPSTSTDQPVTPEPTSRATRGRTNRSSVKTPESIVPIAPELQPSTSRNQLVTPEPTSRATRCRTNRSSVKTPEPVVPTAPEPHPTTSTDQPVTPKLTSRATRRKTNRSSVKTPKPVEPAASDLEPFTPTDQSVTPEAIAQGGQSKTLRSSTVRAMPVPTTPEFQSPVTTDQPISPEPITQPSCIKRQRAAGNPGSLAAPIDHKPCSAPLEPKSQASRNQRWGAVRAAESLTAIPEPASPQLLETPIHASQIQKVEPAGRSRFTPELQPKASQSRKRSLATMDSPPHQKQPQRGEVSQKTVIIKEEEEDTAEKPGKEEDVVTPKPGKRKRDQAEEEPNRIPSRSLRRTKLNQESTAPKVLFTGVVDARGERAVLALGGSLAGSAAEASHLVTDRIRRTVKFLCALGRGIPILSLDWLHQSRKAGFFLPPDEYVVTDPEQEKNFGFSLQDALSRARERRLLEGYEIYVTPGVQPPPPQMGEIISCCGGTYLPSMPRSYKPQRVVITCPQDFPHCSIPLRVGLPLLSPEFLLTGVLKQEAKPEAFVLSPLEMSST
...,...,...,...
3846,Q96EY8,LLPS-,MAVCGLGSRLGLGSRLGLRGCFGAARLLYPRFQSRGPQGVEDGDRPQPSSKTPRIPKIYTKTGDKGFSSTFTGERRPKDDQVFEAVGTTDELSSAIGFALELVTEKGHTFAEELQKIQCTLQDVGSALATPCSSAREAHLKYTTFKAGPILELEQWIDKYTSQLPPLTAFILPSGGKISSALHFCRAVCRRAERRVVPLVQMGETDANVAKFLNRLSDYLFTLARYAAMKEGNQEKIYMKNDPSAESEGL
3847,Q96K19,LLPS-,MAKYQGEVQSLKLDDDSVIEGVSDQVLVAVVVSFALIATLVYALFRNVHQNIHPENQELVRVLREQLQTEQDAPAATRQQFYTDMYCPICLHQASFPVETNCGHLFCGACIIAYWRYGSWLGAISCPICRQTVTLLLTVFGEDDQSQDVLRLHQDINDYNRRFSGQPRSIMERIMDLPTLLRHAFREMFSVGGLFWMFRIRIILCLMGAFFYLISPLDFVPEALFGILGFLDDFFVIFLLLIYISIMYREVITQRLTR
3848,Q02080,LLPS-,MGRKKIQISRILDQRNRQVTFTKRKFGLMKKAYELSVLCDCEIALIIFNSANRLFQYASTDMDRVLLKYTEYSEPHESRTNTDILETLKRRGIGLDGPELEPDEGPEEPGEKFRRLAGEGGDPALPRPRLYPAAPAMPSPDVVYGALPPPGCDPSGLGEALPAQSRPSPFRPAAPKAGPPGLVHPLFSPSHLTSKTPPPLYLPTEGRRSDLPGGLAGPRGGLNTSRSLYSGLQNPCSTATPGPPLGSFPFLPGG

In [65]:
def sample_sequences_for_validation(df, n=20):
    """
    Samples n random proteins from the DataFrame and displays their UniProt IDs and sequences
    for validation purposes.
    
    Parameters:
    - df: DataFrame with at least 'uniprot_id' and 'sequence' columns
    - n: Number of samples to display (default: 5)
    """
    # Make sure we don't try to sample more rows than exist
    sample_size = min(n, len(df))
    
    # Sample random rows
    samples = df.sample(n=sample_size, random_state=42)  # Using random_state for reproducibility
    
    print(f"\n===== {sample_size} Random Proteins for Validation =====\n")
    
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        uniprot_id = row['uniprot_id']
        category = row['Category']
        
        # Get sequence and truncate if too long for display
        sequence = row['sequence']
        display_seq = sequence
        
        print(f"Sample {i}:")
        print(f"UniProt ID: {uniprot_id}")
        print(f"Category: {category}")
        print(f"Sequence (first 50 chars): {display_seq}")
        print(f"Sequence length: {len(sequence)} amino acids")
        print("-" * 60)
    
    return samples

# Example usage
validation_samples = sample_sequences_for_validation(df_clean, n=20)


===== 20 Random Proteins for Validation =====

Sample 1:
UniProt ID: O94832
Category: LLPS+
Sequence (first 50 chars): MAEQESLEFGKADFVLMDTVSMPEFMANLRLRFEKGRIYTFIGEVVVSVNPYKLLNIYGRDTIEQYKGRELYERPPHLFAIADAAYKAMKRRSKDTCIVISGESGAGKTEASKYIMQYIAAITNPSQRAEVERVKNMLLKSNCVLEAFGNAKTNRNDNSSRFGKYMDINFDFKGDPIGGHINNYLLEKSRVIVQQPGERSFHSFYQLLQGGSEQMLRSLHLQKSLSSYNYIHVGAQLKSSINDAAEFRVVADAMKVIGFKPEEIQTVYKILAAILHLGNLKFVVDGDTPLIENGKVVSIIAELLSTKTDMVEKALLYRTVATGRDIIDKQHTEQEASYGRDAFAKAIYERLFCWIVTRINDIIEVKNYDTTIHGKNTVIGVLDIYGFEIFDNNSFEQFCINYCNEKLQQLFIQLVLKQEQEEYQREGIPWKHIDYFNNQIIVDLVEQQHKGIIAILDDACMNVGKVTDEMFLEALNSKLGKHAHFSSRKLCASDKILEFDRDFRIRHYAGDVVYSVIGFIDKNKDTLFQDFKRLMYNSSNPVLKNMWPEGKLSITEVTKRPLTAATLFKNSMIALVDNLASKEPYYVRCIKPNDKKSPQIFDDERCRHQVEYLGLLENVRVRRAGFAFRQTYEKFLHRYKMISEFTWPNHDLPSDKEAVKKLIERCGFQDDVAYGKTKIFIRTPRTLFTLEELRAQMLIRIVLFLQKVWRGTLARMRYKRTKAALTIIRYYRRYKVKSYIHEVARRFHGVKTMRDYGKHVKWPSPPKVLRRFEEALQTIFNRWRASQLIKSIPASDLPQVRAKVAAVEMLKGQRADLGLQRAWEGNYLASKPDTPQTSGTFVPVANELKRKDKYMNVLFSCHVRKVNRFSKVEDRAIFVT

In [66]:
['delta_5_HB_RS', 'Uniprot_ID', 'delta_5_HB', 'LCR_sequence', 'Category', 'Sequence', 'delta_5_HB']

['delta_5_HB_RS',
 'Uniprot_ID',
 'delta_5_HB',
 'LCR_sequence',
 'Category',
 'Sequence',
 'delta_5_HB']

In [67]:
df_clean.columns = ['Uniprot_ID', 'Category', 'Sequence']
df_clean['reshuffled_sequence'] = None
df_clean['delta_5_HB_RS'] = None
df_clean['delta_5_HB'] = None
df_clean['LCR_sequence'] = None

In [68]:
df_clean.columns

Index(['Uniprot_ID', 'Category', 'Sequence', 'reshuffled_sequence',
       'delta_5_HB_RS', 'delta_5_HB', 'LCR_sequence'],
      dtype='object')

In [69]:
# read original feature file

In [70]:
combined = pd.read_csv('src/files/DeePhase/Paper_code/Features/training_data_features.csv')

In [71]:
PDB_star = combined[~combined['Category'].isin(["LLPS+","LLPS-"])]
PDB_star_filter = PDB_star[~PDB_star["Uniprot_ID"].isin(df_clean["Uniprot_ID"])]

In [72]:
PDB_star_filter.columns

Index(['Sequence', 'Uniprot_ID', 'Category', 'Sequence_length', 'LCR_frac',
       'LCR_sequence', 'LCR_length', 'Hydrophobicity', 'Shannon_entropy',
       'IDR_frac',
       ...
       '190', '191', '192', '193', '194', '195', '196', '197', '198', '199'],
      dtype='object', length=274)

In [73]:
PDB_star_filter

,Sequence,Uniprot_ID,Category,Sequence_length,LCR_frac,LCR_sequence,LCR_length,Hydrophobicity,Shannon_entropy,IDR_frac,...,190,191,192,193,194,195,196,197,198,199
221,MPKPGILKSKSMFCVIYRSSKRDQTYLYVEKKDDFSRVPEELMKGFGQPQLAMILPLDGRKKLVNADIEKVKQALTEQGYYLQLPPPPEDLLKQHLSVMGQKTDDTNK,P0AB44,PDB*,108,0.000000,NaN,0,-72.3,3.989090,0.000,...,-9.662743,8.703130,0.521223,3.522244,-6.771785,-11.627868,-1.847447,9.816297,2.999661,-11.347100
222,MAEYGTLLQDLTNNITLEDLEQLKSACKEDIPSEKSEEITTGSAWFSFLESHNKLDKDNLSYIEHIFEISRRPDLLTMVVDYRTRVLKISEEDELDTKLTRIPSAKKYKDIIRQPSEEEIIKLAPPPKKA,Q9Z297,PDB*,130,0.000000,NaN,0,-83.7,3.941855,0.000,...,-11.139148,9.114004,-0.146163,3.876888,-8.207906,-12.894430,-2.863589,10.664391,5.379802,-14.025694
223,MKISDGNWLIQPGLNLIHPLQVFEVEQQDNEMVVYAAPRDVRERTWQLDTPLFTLRFFSPQEGIVGVRIEHFQGALNNGPHYPLNILQDVKVTIENTERYAEFKSGNLSARVSKGEFWSLDFLRNGERITGSQVKNNGYVQDTNNQRNYMFERLDLGVGETVYGLGERFTALVRNGQTVETWNRDGGTSTEQAYKNIPFYMTNRGYGVLVNHPQCVSFEVGSEKVSKVQFSVESEYLEYFVIDGPTPKAVLDRYTRFTGRPALPPAWSFGLWLTTSFTTNYDEATVNSFIDGMAERNLPLHVFHFDCFWMKAFQWCDFEWDPLTFPDPEGMIRRLKAKGLKICVWINPYIGQKSPVFKELQEKGYLLKRPDGSLWQWDKWQPGLAIYDFTNPDACKWYADKLKGLVAMGVDCFKTDFGERIPTDVQWFDGSDPQKMHNHYAYIYNELVWNVLKDTVGEEEAVLFARSASVGAQKFPVHWGGDCYANYESMAESLRGGLSIGLSGFGFWSHDIGGFENTAPAHVYKRWCAFGLLSSHSRLHGSKSYRVPWAYDDESCDVVRFFTQLKCRMMPYLYREAARANARGTPMMRAMMMEFPDDPACDYLDRQYMLGDNVMVAPVFTEAGDVQFYLPEGRWTHLWHNDELDGSRWHKQQHGFLSLPVYVRDNTLLALGNNDQRPDYVWHEGTAFHLFNLQDGHEAVCEVPAADGSVIFTLKAARTGNTITVTGAGEAKNWTLCLRNVVKVNGLQDGSQAESEQGLVVKPQGNALTITL,P31434,PDB*,772,0.036269,ESLRGGLSIGLSGFFGLLSSHSRLHGSK,28,-306.3,4.222127,0.000,...,-96.632830,68.685610,-1.571157,-0.578935,-37.655746,-76.653720,-21.485865,58.357730,14.780958,-81.916030
224,MKVRVKAPCTSANLGVGFDVFGLCLKEPYDVIEVEAIDDKEIIIEVDDKNIPTDPDKNVAGIVAKKMIDDFNIGKGVKITIKKGVKAGSGLGSSAASSAGTAYAINELFKLNLDKLKLVDYASYGELASSGAKHADNVAPAIFGGFTMVTNYEPLEVLHIPIDFKLDILIAIPNISINTKEAREILPKAVGLKDLVNNVGKACGMVYALYNKDKSLFGRYMMSDKVIEPVRGKLIPNYFKIKEEVKDKVYGITISGSGPSIIAFPKEEFIDEVENILRDYYENTIRTEVGKGVEVV,Q58504,PDB*,296,0.209459,YDVIEVEAIDDKEIIIEVDDNIGKGVKITIKKGVKAGSGLGSSAASSAGTYGITISGSGPSI,62,1.4,3.926290,0.000,...,-27.035944,27.113945,8.554350,5.890775,-15.802848,-34.797222,-4.581340,25.997040,11.849493,-30.570827
225,MFTAKLIKGKTYNVMGITFRAGVSQTVPKKLYEYLNENPYFILTQELNNQKDDPINYTESELKGMNKAEHESIISNLGRNPSDFKNADERIAYILKQIDNKGE,P45922,PDB*,103,0.000000,NaN,0,-76.5,3.999228,0.000,...,-9.160376,9.021527,2.431501,3.892605,-7.569968,-10.288531,-2.254363,8.737546,4.101063,-8.854371
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353,MAYSEKVIDHYENPRNVGSLDKKDSNVGTGMVGAPACGDVMQLQIKVDDNGIIEDAKFKTYGCGSAIASSSLITEWVKGKSLEEAGAIKNSQIAEELELPPVKVHCSILAEDAIKAAIADYKAKQG,Q57074,PDB*,126,0.119048,LAEDAIKAAIADYKA,15,-29.7,3.978553,0.000,...,-11.353363,10.575824,-1.060864,0.530808,-8.402147,-12.559664,-1.845978,9.876379,5.892338,-13.287112
354,MANKAVNDFILAMNYDKKKLLTHQGESIENRFIKEGNQLPDEFVVIERKKRSLSTNTSDISVTATNDSRLYPGALLVVDETLLENNPTLLAVDRAPMTYSIDLPGLASSDSFLQVEDPSNSSVRGAVNDLLAKWHQDYGQVNNVPARMQYEKITAHSMEQLKVKFGSDFEKTGNSLDIDFNSVHSGEKQIQIVNFKQIYYTVSVDAVKNPGDVFQDTVTVEDLKQRGISAERPLVYISSVAYGRQVYLKLETTSKSDEVEAAFEALIKGVKVAPQTEWKQILDNTEVKAVILGGDPSSGARVVTGKVDMVEDLIQEGSRFTADHPGLPISYTTSFLRDNVVATFQNSTDYVETKVTAYRNGDLLLDHSGAYVAQYYITWDELSYDHQGKEVLTPKAWDRNGQDLTAHFTTSIPLKGNVRNLSVKIRECTGLAWEWWRTVYEKTDLPLVRKRTISIWGTTLYPQVEDKVEND,Q7ZAK5,PDB*,471,0.000000,NaN,0,-197.6,4.106212,0.000,...,-50.752130,38.518852,3.580008,5.656966,-28.760092,-47.348286,-6.772511,32.666420,17.956667,-50.716500
355,MKVLFLTANEFEDVELIYPYHRLKEEGHEVYIASFERGTITGKHGYSVKVDLTFDKVNPEEFDALVLPGGRAPERVRLNEKAVSIARKMFSEGKPVASICHGPQILISAGVLRGRKGTSYPGIKDDMINAGVEWVDAEVVVDGNWVSSRVPADLYAWMREFVKLLK,O59413,PDB*,166,0.000000,NaN,0,-28.2,4.069591,0.000,...,-15.005748,14.802292,5.751145,1.768096,-11.201273,-12.021826,-4.200112,14.085509,3.769198,-18.857725
356,MKVLLVLTDAYSDCEKAITYAVNFSEKLGAELDILAVLEDVYNLERANVTFGLPFPPEIKEESKKRIERRLREVWEKLTGSTEIPGVEYRIGPLSEEVKKFVEGKGYELVVWACYPSAYLCKVIDGLNLASLIVK,O66565,PDB*,135,0.118519,PEIKEESKKRIERRLR,16,-1

In [75]:
# combine PDB_star_filter and df_clean
# match the column names
import pandas as pd
import numpy as np


# Step 3: Ensure column order is consistent
PDB_star_filter = PDB_star_filter[['Uniprot_ID', 'Category', 'Sequence']]
df_clean = df_clean[['Uniprot_ID', 'Category', 'Sequence']]

# Step 4: Combine the DataFrames
combined_df = pd.concat([PDB_star_filter, df_clean], ignore_index=True)

In [76]:
combined_df.columns

Index(['Uniprot_ID', 'Category', 'Sequence'], dtype='object')

In [77]:
combined_df

,Uniprot_ID,Category,Sequence
0,P0AB44,PDB*,MPKPGILKSKSMFCVIYRSSKRDQTYLYVEKKDDFSRVPEELMKGFGQPQLAMILPLDGRKKLVNADIEKVKQALTEQGYYLQLPPPPEDLLKQHLSVMGQKTDDTNK
1,Q9Z297,PDB*,MAEYGTLLQDLTNNITLEDLEQLKSACKEDIPSEKSEEITTGSAWFSFLESHNKLDKDNLSYIEHIFEISRRPDLLTMVVDYRTRVLKISEEDELDTKLTRIPSAKKYKDIIRQPSEEEIIKLAPPPKKA
2,P31434,PDB*,MKISDGNWLIQPGLNLIHPLQVFEVEQQDNEMVVYAAPRDVRERTWQLDTPLFTLRFFSPQEGIVGVRIEHFQGALNNGPHYPLNILQDVKVTIENTERYAEFKSGNLSARVSKGEFWSLDFLRNGERITGSQVKNNGYVQDTNNQRNYMFERLDLGVGETVYGLGERFTALVRNGQTVETWNRDGGTSTEQAYKNIPFYMTNRGYGVLVNHPQCVSFEVGSEKVSKVQFSVESEYLEYFVIDGPTPKAVLDRYTRFTGRPALPPAWSFGLWLTTSFTTNYDEATVNSFIDGMAERNLPLHVFHFDCFWMKAFQWCDFEWDPLTFPDPEGMIRRLKAKGLKICVWINPYIGQKSPVFKELQEKGYLLKRPDGSLWQWDKWQPGLAIYDFTNPDACKWYADKLKGLVAMGVDCFKTDFGERIPTDVQWFDGSDPQKMHNHYAYIYNELVWNVLKDTVGEEEAVLFARSASVGAQKFPVHWGGDCYANYESMAESLRGGLSIGLSGFGFWSHDIGGFENTAPAHVYKRWCAFGLLSSHSRLHGSKSYRVPWAYDDESCDVVRFFTQLKCRMMPYLYREAARANARGTPMMRAMMMEFPDDPACDYLDRQYMLGDNVMVAPVFTEAGDVQFYLPEGRWTHLWHNDELDGSRWHKQQHGFLSLPVYVRDNTLLALGNNDQRPDYVWHEGTAFHLFNLQDGHEAVCEVPAADGSVIFTLKAARTGNTITVTGAGEAKNWTLCLRNVVKVNGLQDGSQAESEQGLVVKPQGNALTITL
3,Q58504,PDB*,MKVRVKAPCTSANLGVGFDVFGLCLKEPYDVIEVEAIDDKEIIIEVDDKNIPTDPDKNVAGIVAKKMIDDFNIGKGVKITIKKGVKAGSGLGSSAASSAGTAYAINELFKLNLDKLKLVDYASYGELASSGAKHADNVAPAIFGGFTMVTNYEPLEVLHIPIDFKLDILIAIPNISINTKEAREILPKAVGLKDLVNNVGKACGMVYALYNKDKSLFGRYMMSDKVIEPVRGKLIPNYFKIKEEVKDKVYGITISGSGPSIIAFPKEEFIDEVENILRDYYENTIRTEVGKGVEVV
4,P45922,PDB*,MFTAKLIKGKTYNVMGITFRAGVSQTVPKKLYEYLNENPYFILTQELNNQKDDPINYTESELKGMNKAEHESIISNLGRNPSDFKNADERIAYILKQIDNKGE
...,...,...,...
3977,Q96EY8,LLPS-,MAVCGLGSRLGLGSRLGLRGCFGAARLLYPRFQSRGPQGVEDGDRPQPSSKTPRIPKIYTKTGDKGFSSTFTGERRPKDDQVFEAVGTTDELSSAIGFALELVTEKGHTFAEELQKIQCTLQDVGSALATPCSSAREAHLKYTTFKAGPILELEQWIDKYTSQLPPLTAFILPSGGKISSALHFCRAVCRRAERRVVPLVQMGETDANVAKFLNRLSDYLFTLARYAAMKEGNQEKIYMKNDPSAESEGL
3978,Q96K19,LLPS-,MAKYQGEVQSLKLDDDSVIEGVSDQVLVAVVVSFALIATLVYALFRNVHQNIHPENQELVRVLREQLQTEQDAPAATRQQFYTDMYCPICLHQASFPVETNCGHLFCGACIIAYWRYGSWLGAISCPICRQTVTLLLTVFGEDDQSQDVLRLHQDINDYNRRFSGQPRSIMERIMDLPTLLRHAFREMFSVGGLFWMFRIRIILCLMGAFFYLISPLDFVPEALFGILGFLDDFFVIFLLLIYISIMYREVITQRLTR
3979,Q02080,LLPS-,MGRKKIQISRILDQRNRQVTFTKRKFGLMKKAYELSVLCDCEIALIIFNSANRLFQYASTDMDRVLLKYTEYSEPHESRTNTDILETLKRRGIGLDGPELEPDEGPEEPGEKFRRLAGEGGDPALPRPRLYPAAPAMPSPDVVYGALPPPGCDPSGLGEALPAQSRPSPFRPAAPKAGPPGLVHPLFSPSHLTSKTPPPLYLPTEGRRSDLPGGLAGPRGGLNTSRSLYSGLQNPCSTATPGPPLGSFPFLPGGPPVGAEAWARRVPQPAAPPRRPPQSASSLSASLRPPGAPATFLRPSPIPCSSPGPWQSLCGLGPPCAGCPWPTAGPGRRSPGGTSPERSPGTARARGDPTSLQASSEKTQQ
3980,Q07654,LLPS-,MAARALCMLGLVLALLSSSSAEEYVGLSANQCAVPAKDRVDCGYPHVTPKECNNRGCCFDSRIPGVPWCFKPLQEAECTF


In [78]:
for idx, row in combined_df.iterrows():
    sequence = row['Sequence']
    feature_df = extract_seq_feature(sequence)  # This should return a 1-row DataFrame
    
    for col in feature_df.columns:
        combined_df.at[idx, col] = feature_df.at[0, col]

C:\Users\LuVul\AppData\Local\Temp\ipykernel_22520\2424988570.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined_df.at[idx, col] = feature_df.at[0, col]
C:\Users\LuVul\AppData\Local\Temp\ipykernel_22520\2424988570.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined_df.at[idx, col] = feature_df.at[0, col]
C:\Users\LuVul\AppData\Local\Temp\ipykernel_22520\2424988570.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [79]:
combined_df

,Uniprot_ID,Category,Sequence,sequence_final,Sequence_length,LCR_frac,LCR_length,Hydrophobicity,Shannon_entropy,IDR_frac,...,190,191,192,193,194,195,196,197,198,199
0,P0AB44,PDB*,MPKPGILKSKSMFCVIYRSSKRDQTYLYVEKKDDFSRVPEELMKGFGQPQLAMILPLDGRKKLVNADIEKVKQALTEQGYYLQLPPPPEDLLKQHLSVMGQKTDDTNK,MPKPGILKSKSMFCVIYRSSKRDQTYLYVEKKDDFSRVPEELMKGFGQPQLAMILPLDGRKKLVNADIEKVKQALTEQGYYLQLPPPPEDLLKQHLSVMGQKTDDTNK,108.0,0.000000,0.0,-72.3,3.989090,0.0,...,-9.662743,8.703130,0.521223,3.522244,-6.771785,-11.627868,-1.847447,9.816297,2.999661,-11.347100
1,Q9Z297,PDB*,MAEYGTLLQDLTNNITLEDLEQLKSACKEDIPSEKSEEITTGSAWFSFLESHNKLDKDNLSYIEHIFEISRRPDLLTMVVDYRTRVLKISEEDELDTKLTRIPSAKKYKDIIRQPSEEEIIKLAPPPKKA,MAEYGTLLQDLTNNITLEDLEQLKSACKEDIPSEKSEEITTGSAWFSFLESHNKLDKDNLSYIEHIFEISRRPDLLTMVVDYRTRVLKISEEDELDTKLTRIPSAKKYKDIIRQPSEEEIIKLAPPPKKA,130.0,0.000000,0.0,-83.7,3.941855,0.0,...,-11.139148,9.114004,-0.146163,3.876888,-8.207906,-12.894430,-2.863589,10.664391,5.379802,-14.025694
2,P31434,PDB*,MKISDGNWLIQPGLNLIHPLQVFEVEQQDNEMVVYAAPRDVRERTWQLDTPLFTLRFFSPQEGIVGVRIEHFQGALNNGPHYPLNILQDVKVTIENTERYAEFKSGNLSARVSKGEFWSLDFLRNGERITGSQVKNNGYVQDTNNQRNYMFERLDLGVGETVYGLGERFTALVRNGQTVETWNRDGGTSTEQAYKNIPFYMTNRGYGVLVNHPQCVSFEVGSEKVSKVQFSVESEYLEYFVIDGPTPKAVLDRYTRFTGRPALPPAWSFGLWLTTSFTTNYDEATVNSFIDGMAERNLPLHVFHFDCFWMKAFQWCDFEWDPLTFPDPEGMIRRLKAKGLKICVWINPYIGQKSPVFKELQEKGYLLKRPDGSLWQWDKWQPGLAIYDFTNPDACKWYADKLKGLVAMGVDCFKTDFGERIPTDVQWFDGSDPQKMHNHYAYIYNELVWNVLKDTVGEEEAVLFARSASVGAQKFPVHWGGDCYANYESMAESLRGGLSIGLSGFGFWSHDIGGFENTAPAHVYKRWCAFGLLSSHSRLHGSKSYRVPWAYDDESCDVVRFFTQLKCRMMPYLYREAARANARGTPMMRAMMMEFPDDPACDYLDRQYMLGDNVMVAPVFTEAGDVQFYLPEGRWTHLWHNDELDGSRWHKQQHGFLSLPVYVRDNTLLALGNNDQRPDYVWHEGTAFHLFNLQDGHEAVCEVPAADGSVIFTLKAARTGNTITVTGAGEAKNWTLCLRNVVKVNGLQDGSQAESEQGLVVKPQGNALTITL,MKISDGNWLIQPGLNLIHPLQVFEVEQQDNEMVVYAAPRDVRERTWQLDTPLFTLRFFSPQEGIVGVRIEHFQGALNNGPHYPLNILQDVKVTIENTERYAEFKSGNLSARVSKGEFWSLDFLRNGERITGSQVKNNGYVQDTNNQRNYMFERLDLGVGETVYGLGERFTALVRNGQTVETWNRDGGTSTEQAYKNIPFYMTNRGYGVLVNHPQCVSFEVGSEKVSKVQFSVESEYLEYFVIDGPTPKAVLDRYTRFTGRPALPPAWSFGLWLTTSFTTNYDEATVNSFIDGMAERNLPLHVFHFDCFWMKAFQWCDFEWDPLTFPDPEGMIRRLKAKGLKICVWINPYIGQKSPVFKELQEKGYLLKRPDGSLWQWDKWQPGLAIYDFTNPDACKWYADKLKGLVAMGVDCFKTDFGERIPTDVQWFDGSDPQKMHNHYAYIYNELVWNVLKDTVGEEEAVLFARSASVGAQKFPVHWGGDCYANYESMAESLRGGLSIGLSGFGFWSHDIGGFENTAPAHVYKRWCAFGLLSSHSRLHGSKSYRVPWAYDDESCDVVRFFTQLKCRMMPYLYREAARANARGTPMMRAMMMEFPDDPACDYLDRQYMLGDNVMVAPVFTEAGDVQFYLPEGRWTHLWHNDELDGSRWHKQQHGFLSLPVYVRDNTLLALGNNDQRPDYVWHEGTAFHLFNLQDGHEAVCEVPAADGSVIFTLKAARTGNTITVTGAGEAKNWTLCLRNVVKVNGLQDGSQAESEQGLVVKPQGNALTITL,772.0,0.036269,28.0,-306.3,4.222127,0.0,...,-96.632828,68.685608,-1.571157,-0.578935,-37.655746,-76.653717,-21.485865,58.357731,14.780958,-81.916031
3,Q58504,PDB*,MKVRVKAPCTSANLGVGFDVFGLCLKEPYDVIEVEAIDDKEIIIEVDDKNIPTDPDKNVAGIVAKKMIDDFNIGKGVKITIKKGVKAGSGLGSSAASSAGTAYAINELFKLNLDKLKLVDYASYGELASSGAKHADNVAPAIFGGFTMVTNYEPLEVLHIPIDFKLDILIAIPNISINTKEAREILPKAVGLKDLVNNVGKACGMVYALYNKDKSLFGRYMMSDKVIEPVRGKLIPNYFKIKEEVKDKVYGITISGSGPSIIAFPKEEFIDEVENILRDYYENTIRTEVGKGVEVV,MKVRVKAPCTSANLGVGFDVFGLCLKEPYDVIEVEAIDDKEIIIEVDDKNIPTDPDKNVAGIVAKKMIDDFNIGKGVKITIKKGVKAGSGLGSSAASSAGTAYAINELFKLNLDKLKLVDYASYGELASSGAKHADNVAPAIFGGFTMVTNYEPLEVLHIPIDFKLDILIAIPNISINTKEAREILPKAVGLKDLVNNVGKACGMVYALYNKDKSLFGRYMMSDKVIEPVRGKLIPNYFKIKEEVKDKVYGITISGSGPSIIAFPKEEFIDEVENILRDYYENTIRTEVGKGVEVV,296.0,0.209459,62.0,1.4,3.926290,0.0,...,-27.035944,27.113945,8.554350,5.890775,-15.802848,-34.797222,-4.581340,25.997040,11.849493,-30.570827
4,P45922,PDB*,MFTAKLIKGKTYNVMGITFRAGVSQTVPKKLYEYLNENPYFILTQELNNQKDDPINYTESELKGMNKAEHESIISNLGRNPSDFKNADERIAYILKQIDNKGE,MFTAKLIKGKTYNVMGITFRAGVSQTVPKKLYEYLNENPYFILTQELNNQKDDPINYTESELKGMNKAEHESIISNLGRNPSDFKNADERIAYILKQIDNKGE,103.0,0.000000,0.0,-76.5,3.999228,0.0,...,-9.160376,9.021527,2.431501,3.892605,-7.569968,-10.288531,-2.254363,8.737546,4.101063,-8.854371
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3977,Q96EY8,LLPS-,MAVCGLGSRLGLGSRLGLRGCFGAARLLYPRFQSRGPQGVEDGDRPQPSSKTPRIPKIYTKTGDKGFSSTFTGERRPKDDQVFEAVGTTDELSSAIGFALELVTEKGHTFAEELQKIQCT

In [80]:
combined_df['reshuffled_sequence'] = None
combined_df['delta_5_HB_RS'] = None
combined_df['delta_5_HB'] = None
combined_df['LCR_sequence'] = None

C:\Users\LuVul\AppData\Local\Temp\ipykernel_22520\1478191683.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined_df['reshuffled_sequence'] = None
C:\Users\LuVul\AppData\Local\Temp\ipykernel_22520\1478191683.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined_df['delta_5_HB_RS'] = None
C:\Users\LuVul\AppData\Local\Temp\ipykernel_22520\1478191683.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining

In [81]:
combined_df.columns = [str(col) for col in combined_df.columns]

In [82]:
combined_df.columns

Index(['Uniprot_ID', 'Category', 'Sequence', 'sequence_final',
       'Sequence_length', 'LCR_frac', 'LCR_length', 'Hydrophobicity',
       'Shannon_entropy', 'IDR_frac',
       ...
       '194', '195', '196', '197', '198', '199', 'reshuffled_sequence',
       'delta_5_HB_RS', 'delta_5_HB', 'LCR_sequence'],
      dtype='object', length=275)

In [83]:
desired_order = combined.columns

In [84]:
combined_df = combined_df[desired_order]

In [85]:
combined_df.to_csv('src/files/DeePhase/Paper_code/Features/picnic_training_data_features.csv')

In [ ]:
def feature_extraction(df, output_filename):
    # extract feature from a df, utilize the deephase function in deephase_utils.py
    for row in df.iterrow():
        seq = row["row"]
        # extract feature here
        # save the feature combine with the original df

    # save df into a file
    df.to_csv(output_filename)
    return df

In [ ]:
def train_model(df_training, training_categories, num_trials, model_name, clf):
    df_training = df_training[df_training['Category'].isin(training_categories)]
    df_training['Category'] = df_training['Category'].map({str(training_categories[0]): 1, training_categories[1]: 0})
    df_training = df_training.reset_index(drop=True)
    df_training = df_training.reindex(sorted(df_training.columns), axis=1)

    gss = GroupShuffleSplit(n_splits = num_trials, test_size = 0.1, random_state=42)
    
    y = df_training['Category']
    groups = df_training['Uniprot_ID']
    X = df_training.drop(columns={'Category', 'Uniprot_ID', 'Sequence'})
    
    accuracy = []; precision = []
    recall = []; roc_auc = []
    for train_index, test_index in gss.split(X, y, groups=groups):
        clf.fit(X.loc[train_index], y.loc[train_index])
        score = clf.score(X.loc[test_index], y.loc[test_index])
        accuracy.append(accuracy_score(clf.predict(X.loc[test_index]), y.loc[test_index]))
        precision.append(precision_score(clf.predict(X.loc[test_index]), y.loc[test_index]))
        recall.append(recall_score(clf.predict(X.loc[test_index]), y.loc[test_index]))
        roc_auc.append(roc_auc_score(y.loc[test_index], clf.predict_proba(X.loc[test_index])[:,1]))
    print('Random forest model: accuracy =' + str(sum(accuracy)/num_trials) + '+/-' + str(np.std(accuracy)/np.sqrt(num_trials)))
    print('Random forest model: precision =' + str(sum(precision)/num_trials) + '+/-' + str(np.std(precision)/np.sqrt(num_trials)))
    print('Random forest model: recall =' + str(sum(recall)/num_trials) + '+/-' + str(np.std(recall)/np.sqrt(num_trials)))
    print('Random forest model: ROC-AUC =' + str(sum(roc_auc)/num_trials) + '+/-' + str(np.std(roc_auc)/np.sqrt(num_trials)))
    
    clf.fit(X, y)
    pickle.dump(clf, open('Models/' + str(model_name) + '.sav', 'wb'))